In [1]:
import os
import joblib
import optuna
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef, roc_auc_score,
    confusion_matrix, make_scorer
)

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
import lightgbm as lgb


In [2]:
df = pd.read_pickle("FingerprintsAll.csv")
df = df.sample(frac=1).reset_index(drop=True)
df

,Name,fp_MACCS,fp_Morgan_512B,fp_Morgan_512B_r3,fp_Morgan_1024B,fp_Morgan_1024B_r3,fp_MAP4_512B,fp_MAP4_512B_r3,fp_MAP4_1024B,fp_MAP4_1024B_r3,Toxicity
0,DB09081,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, ...","[0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, ...","[0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, ...",0
1,T3DB4504,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...","[1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, ...","[1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, ...","[1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, ...",1
2,D03930,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, ...","[0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, ...",0
3,DB00116,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, ...","[1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, ...","[1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, ...",0
4,T3DB0135,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, ...","[1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, ...","[0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, ...","[0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, ...",1
...,...,...,...,...,...,...,...,...,...,...,...
6398,ZINC00895113,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, ...","[0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, ...","[0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, ...","[0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, ...",1
6399,ZINC05975205,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, ...","[0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, ...","[0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, ...","[0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, ...",1
6400,D05626,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...","[1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, ...","[1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, ...","[0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 

In [3]:
#By using Morgan fingerprints with radius of 3 and 1024 bits
X = np.array(list((df['fp_MACCS']))).astype(float)
#X.shape
y = df["Toxicity"].values
#y.shape
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=45767)

# Print the shape of training and testing data
print("Shape of training data:", X_train.shape)
print("Shape of test data:", X_test.shape)
# NBVAL_CHECK_OUTPUT

Shape of training data: (5122, 167)
Shape of test data: (1281, 167)


In [4]:

RANDOM_STATE=789

def baseline_model(model_name):

    if model_name == "RF":
        return RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=8
        )

    elif model_name == "Ridge":
        return RidgeClassifier()

    elif model_name == "KNN":
        return KNeighborsClassifier()

    elif model_name == "SVC":
        return SVC(probability=True)

    elif model_name == "LGBM":
        return lgb.LGBMClassifier(
            random_state=RANDOM_STATE,
            verbosity=-1,
            n_jobs=8
        )


In [5]:
def build_model(trial, model_name):

    if model_name == "RF":
        return RandomForestClassifier(
            n_estimators=trial.suggest_int("n_estimators", 100, 1000),
            #max_depth=trial.suggest_int("max_depth", 3, 20),
            random_state=RANDOM_STATE,
            n_jobs=8
        )

    elif model_name == "Ridge":
        return RidgeClassifier(
            alpha=trial.suggest_float("alpha", 1e-3, 100, log=True)
        )

    elif model_name == "KNN":
        return KNeighborsClassifier(
            n_neighbors=trial.suggest_int("n_neighbors", 3, 25),
            weights=trial.suggest_categorical("weights", ["uniform", "distance"])
        )

    elif model_name == "SVC":
        
        kernel = trial.suggest_categorical("kernel", ["rbf", "linear"])
        params = {
            "C": trial.suggest_categorical(
                "C", [2**i for i in range(-7, 8)]
            ),
            "kernel": kernel,
            "probability": True
        }

        if kernel == "rbf":
            params["gamma"] = trial.suggest_categorical(
                "gamma", [2**i for i in range(-15, 4)]
            )

        return SVC(**params)

    elif model_name == "LGBM":
        return lgb.LGBMClassifier(
            n_estimators=trial.suggest_int("n_estimators", 50, 900),
            learning_rate=trial.suggest_float("learning_rate", 1e-4, 0.2),
            max_depth=trial.suggest_int("max_depth", 3, 12),
            max_bin=trial.suggest_int("max_bin", 150, 300),
            num_leaves= trial.suggest_int("num_leaves", 30, 750),
            random_state=RANDOM_STATE,
            verbosity=-1, 
            n_jobs=8
        )


In [6]:
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn)

# Make scorers compatible with cross_validate
specificity = make_scorer(specificity_score)
npv = make_scorer(npv_score)
mcc_scorer = make_scorer(matthews_corrcoef)

In [7]:
def cv_metrics(model):

    cv = StratifiedKFold(
        n_splits=10, shuffle=True, random_state=RANDOM_STATE
    )

    scoring = {
        "Accuracy": "accuracy",
        "Precision": "precision",
        "Sensitivity": "recall",      # recall = sensitivity
        "Specificity": specificity,
        "F1": "f1",
        "BalancedAcc": "balanced_accuracy",
        "MCC": mcc_scorer,
        "NPV": npv,
        "ROC_AUC": "roc_auc"
    }

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=8,
        return_train_score=False
    )

    # Return results in the specified order
    results = {}
    for metric in ["Accuracy","Precision","Sensitivity","Specificity",
                   "F1","BalancedAcc","MCC","NPV","ROC_AUC"]:
        test_metric = f"test_{metric}"
        results[metric] = (np.mean(scores[test_metric]), np.std(scores[test_metric]))

    return results


In [8]:
def optuna_objective(trial, model_name):

    model = build_model(trial, model_name)
    metrics = cv_metrics(model)

    for m, (mean, std) in metrics.items():
        trial.set_user_attr(f"{m}_mean", mean)
        trial.set_user_attr(f"{m}_std", std)

    return metrics["BalancedAcc"][0]


In [9]:
def evaluate(model, X, y):
    # Predicted labels
    y_pred = model.predict(X)
    
    # Predicted scores for ROC-AUC
    if hasattr(model, "predict_proba"):
        y_scores = model.predict_proba(X)[:, 1]  # probability for positive class
    elif hasattr(model, "decision_function"):
        y_scores = model.decision_function(X)     # continuous scores
    else:
        y_scores = None  # fallback, ROC-AUC won't be calculated

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

    # Metrics
    results = {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred),
        "Sensitivity": recall_score(y, y_pred),
        "Specificity": tn / (tn + fp),
        "F1": f1_score(y, y_pred, zero_division=0),
        "BalancedAcc": balanced_accuracy_score(y, y_pred),
        "MCC": matthews_corrcoef(y, y_pred),
        "NPV": tn / (tn + fn)
    }

    # ROC-AUC
    if y_scores is not None:
        results["ROC_AUC"] = roc_auc_score(y, y_scores)
    else:
        results["ROC_AUC"] = None  # explicitly show None if not computable

    return results


In [10]:
### run Models

models = ["RF", "Ridge", "KNN", "SVC", "LGBM"]

#models = ["SVC", "LGBM"]

baseline_results = []
optimized_results = []
test_results = []          # NEW: store test-set performance
best_models = {}

os.makedirs("FinalModels", exist_ok=True)

for model_name in models:

    print(f"\nBuilding baseline model for {model_name}")

    # ---------- BASELINE (CV on training set) ----------
    base_model = baseline_model(model_name)
    base_metrics = cv_metrics(base_model)

    baseline_results.append({
        "Model": model_name,
        "Type": "Baseline",
        **{k: f"{v[0]:.3f} ± {v[1]:.3f}" for k, v in base_metrics.items()}
    })
    print(f"\nBaseline model balanced accuracy: {base_metrics['BalancedAcc']}")

    joblib.dump(
        base_model,
        f"FinalModels/{model_name}_baseline.joblib"
    )

    print(f"saved a baseline model for {model_name} ")
    
    print(f"\nStarting hyperparameter optimization for {model_name}")

    # ---------- OPTIMIZED (CV on training set) ----------
    study = optuna.create_study(direction="maximize", study_name=model_name)
    study.optimize(lambda t: optuna_objective(t, model_name), n_trials=100)

    trial = study.best_trial

    optimized_results.append({
        "Model": model_name,
        "Type": "Optimized (CV)",
        "Accuracy": f"{trial.user_attrs['Accuracy_mean']:.3f} ± {trial.user_attrs['Accuracy_std']:.3f}",
        "Precision": f"{trial.user_attrs['Precision_mean']:.3f} ± {trial.user_attrs['Precision_std']:.3f}",
        "Sensitivity": f"{trial.user_attrs['Sensitivity_mean']:.3f} ± {trial.user_attrs['Sensitivity_std']:.3f}",
        "Specificity": f"{trial.user_attrs['Specificity_mean']:.3f} ± {trial.user_attrs['Specificity_std']:.3f}",
        "F1": f"{trial.user_attrs['F1_mean']:.3f} ± {trial.user_attrs['F1_std']:.3f}",
        "BalancedAcc": f"{trial.user_attrs['BalancedAcc_mean']:.3f} ± {trial.user_attrs['BalancedAcc_std']:.3f}",
        "MCC": f"{trial.user_attrs['MCC_mean']:.3f} ± {trial.user_attrs['MCC_std']:.3f}",
        "NPV": f"{trial.user_attrs['NPV_mean']:.3f} ± {trial.user_attrs['NPV_std']:.3f}",
        "ROC_AUC": f"{trial.user_attrs['ROC_AUC_mean']:.3f} ± {trial.user_attrs['ROC_AUC_std']:.3f}",
    })

    # ====================================================
    # A) PERFORMANCE ESTIMATION ON HELD-OUT TEST SET
    # ====================================================
    model_eval = build_model(trial, model_name)
    model_eval.fit(X_train, y_train)

    test_metrics = evaluate(model_eval, X_test, y_test)

    test_results.append({
        "Model": model_name,
        "Type": "Test set",
        **{k: f"{v:.3f}" for k, v in test_metrics.items()}
    })

    # ====================================================
    # B) FINAL MODEL FOR DEPLOYMENT (REFIT ON FULL DATA)
    # ====================================================
    final_model = build_model(trial, model_name)
    final_model.fit(X, y)     # <-- FULL DATASET

    best_models[model_name] = final_model
    joblib.dump(
        final_model,
        f"FinalModels/{model_name}_optimized_full.joblib"
    )

    print(f"saved an optimized_full model for {model_name} ")



Building baseline model for RF


[I 2026-04-29 14:46:56,917] A new study created in memory with name: RF



Baseline model balanced accuracy: (0.7584202653440251, 0.013590137500914545)
saved a baseline model for RF 

Starting hyperparameter optimization for RF


[I 2026-04-29 14:46:59,084] Trial 0 finished with value: 0.759966556283221 and parameters: {'n_estimators': 465}. Best is trial 0 with value: 0.759966556283221.
[I 2026-04-29 14:47:01,170] Trial 1 finished with value: 0.7598299602626535 and parameters: {'n_estimators': 474}. Best is trial 0 with value: 0.759966556283221.
[I 2026-04-29 14:47:03,597] Trial 2 finished with value: 0.7616716190079267 and parameters: {'n_estimators': 539}. Best is trial 2 with value: 0.7616716190079267.
[I 2026-04-29 14:47:04,363] Trial 3 finished with value: 0.7585882071698403 and parameters: {'n_estimators': 140}. Best is trial 2 with value: 0.7616716190079267.
[I 2026-04-29 14:47:08,026] Trial 4 finished with value: 0.762326354261845 and parameters: {'n_estimators': 838}. Best is trial 4 with value: 0.762326354261845.
[I 2026-04-29 14:47:11,116] Trial 5 finished with value: 0.763073150467426 and parameters: {'n_estimators': 730}. Best is trial 5 with value: 0.763073150467426.
[I 2026-04-29 14:47:12,975] T

saved an optimized_full model for RF 

Building baseline model for Ridge

Baseline model balanced accuracy: (0.7503744826082198, 0.020742529387724893)
saved a baseline model for Ridge 

Starting hyperparameter optimization for Ridge


...

saved an optimized_full model for SVC 

Building baseline model for LGBM


[I 2026-04-29 15:35:04,249] A new study created in memory with name: LGBM



Baseline model balanced accuracy: (0.77087596957645, 0.012509260472248582)
saved a baseline model for LGBM 

Starting hyperparameter optimization for LGBM


[I 2026-04-29 15:37:20,520] Trial 0 finished with value: 0.7519797105433991 and parameters: {'n_estimators': 462, 'learning_rate': 0.17908030434285718, 'max_depth': 8, 'max_bin': 175, 'num_leaves': 397}. Best is trial 0 with value: 0.7519797105433991.
[I 2026-04-29 15:39:27,171] Trial 1 finished with value: 0.7519722181383877 and parameters: {'n_estimators': 544, 'learning_rate': 0.11714546855356377, 'max_depth': 7, 'max_bin': 246, 'num_leaves': 160}. Best is trial 0 with value: 0.7519797105433991.
[I 2026-04-29 15:40:09,013] Trial 2 finished with value: 0.7632265801283665 and parameters: {'n_estimators': 723, 'learning_rate': 0.13877074058889094, 'max_depth': 4, 'max_bin': 209, 'num_leaves': 30}. Best is trial 2 with value: 0.7632265801283665.
[I 2026-04-29 15:40:35,104] Trial 3 finished with value: 0.7663514313692994 and parameters: {'n_estimators': 129, 'learning_rate': 0.08582892617645656, 'max_depth': 7, 'max_bin': 280, 'num_leaves': 80}. Best is trial 3 with value: 0.766351431369

saved an optimized_full model for LGBM 


In [11]:
df_baseline  = pd.DataFrame(baseline_results)
df_cv_opt    = pd.DataFrame(optimized_results)
df_test      = pd.DataFrame(test_results)
output_file = "Model_Performance_Summary.xlsx"

with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    df_baseline.to_excel(writer, sheet_name="Baseline_CV", index=False)
    df_cv_opt.to_excel(writer, sheet_name="Optimized_CV", index=False)
    df_test.to_excel(writer, sheet_name="Test_Set", index=False)

print(f"All results saved to {output_file}")


All results saved to Model_Performance_Summary.xlsx


In [12]:
df_baseline

,Model,Type,Accuracy,Precision,Sensitivity,Specificity,F1,BalancedAcc,MCC,NPV,ROC_AUC
0,RF,Baseline,0.765 ± 0.013,0.756 ± 0.017,0.699 ± 0.020,0.818 ± 0.016,0.726 ± 0.016,0.758 ± 0.014,0.522 ± 0.027,0.771 ± 0.012,0.824 ± 0.006
1,Ridge,Baseline,0.756 ± 0.020,0.744 ± 0.025,0.694 ± 0.030,0.807 ± 0.022,0.718 ± 0.024,0.750 ± 0.021,0.505 ± 0.041,0.766 ± 0.020,0.823 ± 0.016
2,KNN,Baseline,0.755 ± 0.007,0.731 ± 0.010,0.715 ± 0.013,0.788 ± 0.012,0.723 ± 0.008,0.751 ± 0.007,0.504 ± 0.014,0.774 ± 0.008,0.818 ± 0.011
3,SVC,Baseline,0.787 ± 0.013,0.791 ± 0.019,0.712 ± 0.024,0.847 ± 0.018,0.749 ± 0.017,0.780 ± 0.014,0.567 ± 0.027,0.785 ± 0.013,0.850 ± 0.011
4,LGBM,Baseline,0.777 ± 0.013,0.772 ± 0.021,0.713 ± 0.021,0.829 ± 0.022,0.741 ± 0.014,0.771 ± 0.013,0.547 ± 0.026,0.782 ± 0.012,0.845 ± 0.012


In [13]:
df_cv_opt

,Model,Type,Accuracy,Precision,Sensitivity,Specificity,F1,BalancedAcc,MCC,NPV,ROC_AUC
0,RF,Optimized (CV),0.770 ± 0.014,0.763 ± 0.019,0.704 ± 0.023,0.823 ± 0.018,0.732 ± 0.017,0.764 ± 0.014,0.532 ± 0.028,0.775 ± 0.013,0.825 ± 0.007
1,Ridge,Optimized (CV),0.761 ± 0.018,0.752 ± 0.023,0.695 ± 0.029,0.815 ± 0.021,0.722 ± 0.023,0.755 ± 0.019,0.515 ± 0.038,0.768 ± 0.018,0.823 ± 0.016
2,KNN,Optimized (CV),0.768 ± 0.012,0.765 ± 0.018,0.696 ± 0.017,0.827 ± 0.017,0.729 ± 0.014,0.761 ± 0.012,0.529 ± 0.024,0.771 ± 0.010,0.830 ± 0.011
3,SVC,Optimized (CV),0.787 ± 0.014,0.793 ± 0.022,0.709 ± 0.024,0.850 ± 0.021,0.748 ± 0.017,0.779 ± 0.014,0.568 ± 0.029,0.783 ± 0.013,0.849 ± 0.008
4,LGBM,Optimized (CV),0.784 ± 0.016,0.772 ± 0.022,0.733 ± 0.023,0.825 ± 0.022,0.752 ± 0.018,0.779 ± 0.016,0.562 ± 0.031,0.793 ± 0.014,0.847 ± 0.014


In [14]:
df_test

,Model,Type,Accuracy,Precision,Sensitivity,Specificity,F1,BalancedAcc,MCC,NPV,ROC_AUC
0,RF,Test set,0.779,0.768,0.724,0.824,0.745,0.774,0.551,0.787,0.832
1,Ridge,Test set,0.759,0.741,0.706,0.801,0.723,0.754,0.510,0.772,0.829
2,KNN,Test set,0.763,0.757,0.691,0.821,0.722,0.756,0.517,0.767,0.839
3,SVC,Test set,0.802,0.809,0.727,0.862,0.766,0.795,0.597,0.797,0.850
4,LGBM,Test set,0.785,0.778,0.727,0.832,0.752,0.780,0.564,0.791,0.853
